# BLIP2 + LoRA Finetuning

In [26]:
import torch
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel, PeftConfig
from sklearn.model_selection import train_test_split
import logging
import tempfile
from torch.optim import AdamW
import gc
import logging
import random
import re
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Force GPU usage and memory optimizations
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # Use both GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

In [27]:
# Memory management functions
def clear_memory():
    """Clear CUDA cache and Python garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    logger.info(f"Memory cleared. GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def log_memory_usage():
    """Log current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        max_allocated = torch.cuda.max_memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        logger.info(f"GPU Memory: Current: {allocated:.2f} GB, Peak: {max_allocated:.2f} GB, Reserved: {reserved:.2f} GB")
    
    import psutil
    process = psutil.Process(os.getpid())
    ram_usage = process.memory_info().rss / 1e9
    logger.info(f"RAM Usage: {ram_usage:.2f} GB")

In [28]:
class OptimizedBLIP2(Blip2ForConditionalGeneration):
    def forward(self, **kwargs):
        valid_args = {
            'pixel_values', 'input_ids', 'attention_mask',
            'decoder_input_ids', 'decoder_attention_mask',
            'labels', 'output_attentions', 'output_hidden_states',
            'return_dict', 'use_cache'
        }
        
        filtered_kwargs = {
            k: v.to(self.device) if isinstance(v, torch.Tensor) else v
            for k, v in kwargs.items() 
            if k in valid_args
        }
        
        return super().forward(**filtered_kwargs)

class SafeVQADataset(Dataset):
    def __init__(self, dataframe, processor, image_dir="", max_length=64):
        self.data = dataframe
        self.processor = processor
        self.image_dir = image_dir
        self.max_length = max_length
        self.valid_indices = self._validate_data()
        
    def _validate_data(self):
        valid = []
        for idx in tqdm(range(len(self.data)), desc="Validating data"):
            try:
                path = os.path.join(self.image_dir, self.data.iloc[idx]['path'])
                Image.open(path).convert('RGB')
                valid.append(idx)
            except Exception:
                continue
        logger.info(f"Found {len(valid)} valid samples")
        return valid
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.data.iloc[actual_idx]
        
        try:
            image = Image.open(os.path.join(self.image_dir, row['path'])).convert('RGB')
            
            # Process inputs with strict single-word answers
            inputs = self.processor(
                images=image,
                text=row['generated_question'],
                return_tensors="pt",
                padding='max_length',
                max_length=self.max_length,
                truncation=True
            )
            
            # Force single-word answers
            labels = self.processor(
                text=row['generated_answer'].split()[0],  # Take first word
                return_tensors="pt",
                padding='max_length',
                max_length=3,
                truncation=True
            ).input_ids
            
            return {
                'pixel_values': inputs.pixel_values.squeeze().half(),
                'input_ids': inputs.input_ids.squeeze(),
                'attention_mask': inputs.attention_mask.squeeze(),
                'labels': labels.squeeze()
            }
        except Exception as e:
            logger.error(f"Error processing sample {actual_idx}: {e}")
            return None

In [29]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels': torch.stack([b['labels'] for b in batch])
    }

def train_lora_model(model, train_loader, processor):
    # Log memory before LoRA setup
    log_memory_usage()
    
    # Smaller LoRA config to reduce memory usage
    lora_config = LoraConfig(
        r=4,  # Reduced rank
        lora_alpha=16,  # Reduced alpha
        target_modules=["q", "v"],  # Target fewer modules if needed
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        modules_to_save=["lm_head"]
    )
    
    logger.info("Applying LoRA adapter...")
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # Check memory after LoRA setup
    log_memory_usage()

    # Optimizer with conservative settings
    optimizer = AdamW(
        model.parameters(),
        lr=5e-6,  # Lower learning rate
        weight_decay=0.01, 
        eps=1e-8,
        foreach=False
    )
    
    # Shorter training
    num_epochs = 2  # Reduced epochs
    
    # Scheduler
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=10,
        num_training_steps=len(train_loader)*num_epochs
    )

    # Gradient scaler for mixed precision
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    # Training loop with memory management
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        valid_batches = 0
        
        # Clear memory before epoch
        clear_memory()
        
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch_idx, batch in enumerate(progress):
            if batch is None:
                continue
            
            try:
                # Move batch to device
                inputs = {k: v.to(model.device) for k, v in batch.items()}
                
                # Mixed precision forward pass
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(**inputs)
                    loss = outputs.loss
                
                if torch.isnan(loss) or torch.isinf(loss):
                    logger.warning(f"Bad loss value: {loss.item()}, skipping batch")
                    continue
                
                # Gradient accumulation (every 2 batches)
                loss = loss / 2
                scaler.scale(loss).backward()
                
                # Only update every other batch to save memory
                if (batch_idx + 1) % 2 == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()
                
                total_loss += loss.item() * 2  # Adjust for scaling
                valid_batches += 1
                progress.set_postfix({"loss": f"{loss.item()*2:.4f}"})
                
                # Clean up to prevent memory leaks
                del outputs, loss, inputs
                
                # Periodic memory cleanup (every 10 batches)
                if batch_idx % 10 == 0:
                    clear_memory()

            except Exception as e:
                logger.warning(f"Error in batch {batch_idx}: {str(e)}")
                continue
        
        # Epoch summary
        if valid_batches > 0:
            avg_loss = total_loss / valid_batches
            logger.info(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")
        else:
            logger.warning("No valid batches processed this epoch")

        # Save checkpoint for this epoch (with memory management)
        logger.info("Saving checkpoint...")
        checkpoint_dir = f"/kaggle/working/blip2-lora-epoch-{epoch+1}"
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # Clear memory before saving
        clear_memory()
        
        # Save model weights directly without full materialization
        logger.info("Moving model to CPU for saving...")
        model.to('cpu')
        
        # Save adapter weights only - much smaller
        logger.info(f"Saving adapter weights to {checkpoint_dir}")
        model.save_pretrained(checkpoint_dir)
        
        # Move model back to device
        logger.info("Moving model back to device...")
        model.to(model.device)
        
        # Log memory after checkpoint save
        log_memory_usage()
    
    # Final merge and save with extreme memory care
    logger.info("Preparing for final save...")
    
    # Move model to CPU for final save
    model.to('cpu')
    clear_memory()
    
    # Prepare output directory
    output_dir = "/kaggle/working/blip2-lora-final"
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Save processor first (smaller footprint)
        logger.info(f"Saving processor to {output_dir}")
        processor.save_pretrained(output_dir)
        
        # Save model configuration
        logger.info("Saving model configuration...")
        model.config.save_pretrained(output_dir)
        
        # Save adapter weights instead of full merged model
        adapter_weights_path = os.path.join(output_dir, "adapter_model.bin")
        logger.info(f"Saving adapter weights to {adapter_weights_path}")
        torch.save(model.state_dict(), adapter_weights_path)
        
        # Save adapter config
        adapter_config_path = os.path.join(output_dir, "adapter_config.json")
        logger.info(f"Saving adapter config to {adapter_config_path}")
        model.save_pretrained(output_dir)
        
        logger.info("Model saved successfully without merging")
        
    except Exception as e:
        logger.error(f"Error during model saving: {e}")
        
    return model


In [30]:
def main(image_dir="/kaggle/input/working-vr/abo-images-small/images/small"):
    os.makedirs("/kaggle/working/offload", exist_ok=True)
    
    # Log initial memory state
    log_memory_usage()
    
    # Load processor first - less memory intensive
    logger.info("Loading processor...")
    processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
    
    # Now load model with optimized settings
    logger.info("Loading model...")
    model = OptimizedBLIP2.from_pretrained(
        "Salesforce/blip2-flan-t5-xl",
        torch_dtype=torch.float16,
        device_map="balanced",  # Balance across GPUs
        offload_folder="/kaggle/working/offload",
        offload_state_dict=True,  # Offload weights to reduce GPU memory
        low_cpu_mem_usage=True
    )
    
    # Check GPU usage
    log_memory_usage()
    
    # Load data (in smaller chunks if needed)
    logger.info("Loading dataset...")
    df = pd.read_csv('/kaggle/input/new-data/simplified_vqa_dataset.csv')
    train_df, _ = train_test_split(df, test_size=0.2, random_state=42)
    train_df = train_df.head(18000)
    # Create dataloader with safe batch size
    logger.info("Creating dataset...")
    dataset = SafeVQADataset(train_df, processor, image_dir=image_dir)
    train_loader = DataLoader(
        dataset,
        batch_size=4,  # Increased batch size but still conservative
        collate_fn=collate_fn,
        num_workers=2,  # Use more workers for data loading
        pin_memory=True,
        shuffle=True
    )
    
    # Free up memory before training
    clear_memory()
    
    # Train and save
    logger.info("Starting training...")
    trained_model = train_lora_model(model, train_loader, processor)
    logger.info("Training completed successfully")
    
    # Test inference with fixed function
    try:
        logger.info("Testing inference with the saved model...")
        test_img_path = os.path.join(image_dir, train_df.iloc[0]['path'])
        test_question = train_df.iloc[0]['generated_question']
        answer = infer_fixed(test_img_path, test_question)
        logger.info(f"Test Question: {test_question}")
        logger.info(f"Model Answer: {answer}")
        logger.info(f"Expected Answer: {train_df.iloc[0]['generated_answer'].split()[0]}")
    except Exception as e:
        logger.error(f"Inference test failed: {e}")
    
    return trained_model

In [31]:
main()

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Validating data: 100%|██████████| 18000/18000 [03:15<00:00, 91.88it/s] 


trainable params: 68,157,440 || all params: 4,010,604,032 || trainable%: 1.6994


Epoch 1:   0%|          | 0/4500 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Epoch 2:   0%|          | 0/4500 [00:00<?, ?it/s]huggingface/token

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): OptimizedBLIP2(
      (vision_model): Blip2VisionModel(
        (embeddings): Blip2VisionEmbeddings(
          (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
        )
        (encoder): Blip2Encoder(
          (layers): ModuleList(
            (0-38): 39 x Blip2EncoderLayer(
              (self_attn): Blip2Attention(
                (dropout): Dropout(p=0.0, inplace=False)
                (qkv): Linear(in_features=1408, out_features=4224, bias=True)
                (projection): Linear(in_features=1408, out_features=1408, bias=True)
              )
              (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
              (mlp): Blip2MLP(
                (activation_fn): GELUActivation()
                (fc1): Linear(in_features=1408, out_features=6144, bias=True)
                (fc2): Linear(in_features=6144, out_features=1408, bias=True)
              )
              (la

## Merging the model and pushing It to HuggingFace

In [ ]:
# from huggingface_hub import HfApi, notebook_login
from peft import PeftModel
from transformers import Blip2ForConditionalGeneration, Blip2Processor
import torch

# Login to HF Hub
# notebook_login()  # For notebooks
# OR via CLI: huggingface-cli login

# Load your fine-tuned model
base_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, "/kaggle/input/lora-final/blip2-lora-final")

# Merge adapters for efficient inference
merged_model = model.merge_and_unload()

# Save final version with processor
processor = Blip2Processor.from_pretrained("/kaggle/input/lora-final/blip2-lora-final")
merged_model.save_pretrained("/kaggle/working/final_model")
processor.save_pretrained("/kaggle/working/final_model")

In [ ]:
!ls /kaggle/working/final_model

In [ ]:
!zip -r file.zip /kaggle/working/final_model

In [ ]:
from IPython.display import FileLink
FileLink(r'file.zip')

## Inference code by saved model on Hugging face

In [32]:
!pip install bert_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [33]:
import os
import torch
import logging
import pandas as pd
import re
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import PeftModel
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm
import time
import gc
import warnings
import sys
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [34]:
def preprocess_answer(answer):
    """Enhanced answer preprocessing to improve quality"""
    if not answer or not isinstance(answer, str):
        return ""
    
    # Convert to lowercase and strip
    answer = answer.lower().strip()
    
    # Remove punctuation
    answer = re.sub(r'[^\w\s]', '', answer)
    
    # Remove extra whitespace
    answer = re.sub(r'\s+', ' ', answer).strip()
    
    # Filter out generic single word answers
    if answer in ["a", "an", "the", "of", "in", "on", "is", "are", "am", "i", "it"]:
        return ""
    
    return answer

def compute_bert_score(predictions, references, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    """
    Compute BERTScore between predictions and references
    
    Args:
        predictions: List of prediction strings
        references: List of reference strings
        model_name: Name of the sentence-transformers model to use
    
    Returns:
        Dictionary with precision, recall, and f1 scores
    """
    try:
        # Load model and tokenizer
        logger.info(f"Loading model {model_name} for BERTScore calculation")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
        model.eval()
        
        # Process in batches to avoid OOM
        batch_size = 32
        all_p_scores = []
        all_r_scores = []
        all_f1_scores = []
        
        # Process in batches
        for i in tqdm(range(0, len(predictions), batch_size), desc="Computing BERTScore"):
            batch_pred = predictions[i:i+batch_size]
            batch_ref = references[i:i+batch_size]
            
            # Skip empty strings
            valid_indices = [j for j, (p, r) in enumerate(zip(batch_pred, batch_ref)) 
                            if p.strip() and r.strip()]
            
            if not valid_indices:
                continue
                
            valid_pred = [batch_pred[j] for j in valid_indices]
            valid_ref = [batch_ref[j] for j in valid_indices]
            
            # Tokenize
            pred_inputs = tokenizer(valid_pred, padding=True, truncation=True, 
                                  return_tensors="pt", max_length=128)
            ref_inputs = tokenizer(valid_ref, padding=True, truncation=True, 
                                 return_tensors="pt", max_length=128)
            
            # Move to device
            pred_inputs = {k: v.to(device) for k, v in pred_inputs.items()}
            ref_inputs = {k: v.to(device) for k, v in ref_inputs.items()}
            
            # Get embeddings
            with torch.no_grad():
                pred_outputs = model(**pred_inputs)
                ref_outputs = model(**ref_inputs)
            
            # Mean pooling
            pred_attention_mask = pred_inputs['attention_mask']
            ref_attention_mask = ref_inputs['attention_mask']
            
            pred_embeddings = mean_pooling(pred_outputs, pred_attention_mask)
            ref_embeddings = mean_pooling(ref_outputs, ref_attention_mask)
            
            # Normalize embeddings
            pred_embeddings = torch.nn.functional.normalize(pred_embeddings, p=2, dim=1)
            ref_embeddings = torch.nn.functional.normalize(ref_embeddings, p=2, dim=1)
            
            # Calculate similarity scores
            similarity = torch.matmul(pred_embeddings, ref_embeddings.transpose(0, 1)).diag()
            
            # Add scores
            all_p_scores.extend(similarity.cpu().numpy())
            all_r_scores.extend(similarity.cpu().numpy())  # In BERTScore, P=R for single reference
            
            # Calculate F1 (harmonic mean)
            f1_scores = similarity.cpu().numpy()  # In this case, P=R=F1
            all_f1_scores.extend(f1_scores)
            
        # Clean up
        del model, tokenizer
        torch.cuda.empty_cache()
        
        # Calculate aggregate scores
        if all_p_scores:
            return {
                "precision": np.mean(all_p_scores),
                "recall": np.mean(all_r_scores),
                "f1": np.mean(all_f1_scores),
            }
        else:
            logger.warning("No valid prediction-reference pairs for BERTScore")
            return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
            
    except Exception as e:
        logger.error(f"Error computing BERTScore: {str(e)}")
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

def mean_pooling(model_output, attention_mask):
    """Mean pooling to get sentence embeddings"""
    token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def compute_bleu_score(predictions, references):
    """
    Compute BLEU score between predictions and references
    
    Args:
        predictions: List of prediction strings
        references: List of reference strings
    
    Returns:
        BLEU score
    """
    try:
        # Prepare smoothing function
        smoothie = SmoothingFunction().method1
        
        scores = []
        
        for pred, ref in zip(predictions, references):
            # Tokenize
            tokenized_pred = nltk.word_tokenize(pred.lower())
            tokenized_ref = nltk.word_tokenize(ref.lower())
            
            # Skip empty predictions or references
            if not tokenized_pred or not tokenized_ref:
                continue
                
            # Calculate BLEU with smoothing
            try:
                # Using a list of references (even though we have one)
                score = sentence_bleu([tokenized_ref], tokenized_pred, 
                                     smoothing_function=smoothie,
                                     weights=(0.25, 0.25, 0.25, 0.25))  # Using BLEU-4
                scores.append(score)
            except Exception as e:
                logger.warning(f"Error calculating BLEU for {pred}/{ref}: {str(e)}")
                continue
        
        # Return average BLEU score
        return np.mean(scores) if scores else 0.0
        
    except Exception as e:
        logger.error(f"Error computing BLEU score: {str(e)}")
        return 0.0

def evaluate_with_lenient_matching(predictions, ground_truths):
    """
    Performs lenient matching evaluation that takes into account synonyms and related terms
    
    Args:
        predictions: List of prediction strings
        ground_truths: List of ground truth strings
        
    Returns:
        Dictionary with evaluation metrics
    """
    # Define synonym groups for common terms in your dataset
    synonym_groups = {
        "cellular phone case": ["phone case", "case", "cover", "cellular_phone_case", "phone cover"],
        "yes": ["true", "correct", "affirmative"],
        "no": ["false", "incorrect", "negative"],
        "hardware_hinge": ["hinge", "door hinge"],
        "multicolor": ["multi-color", "multi color", "multicolored", "colorful", "multiple colors"],
        "rectangular": ["rectangle", "rectangular shaped"],
        "blue": ["navy", "navy blue", "azure", "lilac"],
        "gray": ["grey"],
        "brown": ["burgundy", "tan"],
        "teal": ["turquoise", "azure"],
        "gold": ["golden"],
        "hearts": ["heart", "heart-shaped", "heart pattern"],
    }
    
    # Create reverse mapping for quick lookup
    term_to_group = {}
    for main_term, synonyms in synonym_groups.items():
        for synonym in synonyms:
            term_to_group[synonym] = main_term
        term_to_group[main_term] = main_term
    
    # Function to check if terms match using synonyms
    def terms_match(pred, truth):
        # Direct match
        if pred == truth:
            return True
            
        # Synonym match
        pred_group = term_to_group.get(pred)
        truth_group = term_to_group.get(truth)
        
        if pred_group and truth_group and pred_group == truth_group:
            return True
            
        # Partial match (one is substring of another)
        if pred in truth or truth in pred:
            return True
            
        return False
    
    # Count matches
    exact_matches = 0
    lenient_matches = 0
    total = 0
    
    for pred, truth in zip(predictions, ground_truths):
        if not pred or not truth:
            continue
            
        total += 1
        if pred == truth:
            exact_matches += 1
            lenient_matches += 1
        elif terms_match(pred, truth):
            lenient_matches += 1
    
    return {
        "exact_match_rate": exact_matches / total if total > 0 else 0,
        "lenient_match_rate": lenient_matches / total if total > 0 else 0,
        "total_evaluated": total
    }


In [35]:
# Ensure nltk requirements are downloaded
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

logger = logging.getLogger(__name__)


logging.basicConfig(
    level=logging.DEBUG,  # Changed from INFO to DEBUG for more detailed logs
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("vqa_debug.log")
    ]
)
logger = logging.getLogger(__name__)

# Suppress warnings but log them instead
warnings.filterwarnings('ignore')

def get_gpu_info():
    """Get GPU information including count and memory"""
    if not torch.cuda.is_available():
        logger.warning("CUDA is not available, using CPU")
        return 0, []
    
    gpu_count = torch.cuda.device_count()
    gpu_memory = []
    
    for i in range(gpu_count):
        # Get memory stats
        try:
            free_mem = torch.cuda.get_device_properties(i).total_memory
            free_mem = free_mem / (1024 ** 3)  # Convert to GB
            gpu_memory.append(free_mem)
            logger.info(f"GPU {i}: {free_mem:.2f} GB total memory")
            
            # Print current memory usage as well
            allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
            reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)
            logger.info(f"GPU {i}: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")
        except Exception as e:
            logger.error(f"Error getting GPU {i} info: {str(e)}")
            gpu_memory.append(0)
        
    return gpu_count, gpu_memory

def clear_memory(device=None):
    """Clear memory on specific device or all devices"""
    if torch.cuda.is_available():
        try:
            if device is not None:
                with torch.cuda.device(device):
                    torch.cuda.empty_cache()
                    logger.debug(f"Cleared memory on GPU {device}")
            else:
                torch.cuda.empty_cache()
                logger.debug("Cleared memory on all GPUs")
        except Exception as e:
            logger.error(f"Error clearing GPU memory: {str(e)}")
    gc.collect()
    logger.debug("Collected garbage")

def preprocess_answer(answer):
    """Clean and normalize answer text"""
    if not answer or not isinstance(answer, str):
        return ""
    answer = answer.lower()
    answer = re.sub(r'[^\w\s]', '', answer)
    return re.sub(r'\s+', ' ', answer).strip()

In [36]:
class VQAModel:
    def __init__(self, device_id=0):
        """Initialize the VQA model on specified device"""
        self.device_id = device_id

        self.stopwords = set(["a", "an", "the", "is", "are", "am", "i", "you", "he", "she", 
                            "it", "we", "they", "of", "in", "on", "at", "by", "for"])
        
        # Check if CUDA is available
        if torch.cuda.is_available():
            logger.info(f"CUDA is available with {torch.cuda.device_count()} devices")
            self.device = f"cuda:{device_id}"
        else:
            logger.warning("CUDA is not available, falling back to CPU")
            self.device = "cpu"
            
        logger.info(f"Initializing model on {self.device}")
        
        try:
            # Load the processor first (less memory intensive)
            logger.info("Loading BLIP2 processor...")
            
            # Try with more explicit error handling
            try:
                self.processor = Blip2Processor.from_pretrained(
                    "Magneto76/lora_blip2", 
                    local_files_only=False
                )
                logger.info("BLIP2 processor loaded successfully")
            except Exception as e:
                logger.error(f"Failed to load processor from Magneto76/lora_blip2: {str(e)}")
                logger.info("Trying to load processor from base model...")
                
                # Fallback to base model processor
                self.processor = Blip2Processor.from_pretrained(
                    "Salesforce/blip2-flan-t5-xl",
                    local_files_only=False
                )
                logger.info("Loaded processor from base model")
            
            # Now load the model
            logger.info("Loading BLIP2 model...")
            self.model = self._load_model()
            logger.info("Model loaded successfully")
            
        except Exception as e:
            logger.error(f"Model initialization failed on device {device_id}: {str(e)}")
            raise

    def _load_model(self):
        """Load the model with optimized settings"""
        # Free up memory before loading
        clear_memory(self.device_id)
        
        # Set appropriate precision based on GPU
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        logger.info(f"Using dtype: {dtype}")
        
        try:
            # Load base model with optimized memory settings
            logger.info("Loading base BLIP2 model...")
            base_model = Blip2ForConditionalGeneration.from_pretrained(
                "Salesforce/blip2-flan-t5-xl",
                torch_dtype=dtype,
                device_map={"": self.device},
                low_cpu_mem_usage=True
            )
            logger.info("Base model loaded successfully")
            
            # Apply LoRA weights
            logger.info("Applying LoRA weights...")
            try:
                model = PeftModel.from_pretrained(
                    base_model,
                    "Magneto76/lora_blip2",
                    torch_dtype=dtype
                )
                logger.info("LoRA weights applied successfully")
            except Exception as e:
                logger.error(f"Failed to apply LoRA weights: {str(e)}")
                logger.info("Proceeding with base model only")
                model = base_model
            
            return model.eval()
            
        except Exception as e:
            logger.error(f"Error loading model: {str(e)}")
            raise
    
    @torch.inference_mode()
    def infer(self, image_path, question):
        """Run inference on an image with a question"""
        try:
            # Verify the image exists and can be opened
            if not os.path.exists(image_path):
                logger.error(f"Image not found: {image_path}")
                return None
            
            logger.debug(f"Processing image: {image_path}")
            logger.debug(f"Question: {question}")
                
            # Load and process the image
            try:
                image = Image.open(image_path).convert('RGB')
                logger.debug(f"Image loaded successfully: {image.size}")
            except Exception as e:
                logger.error(f"Failed to load image {image_path}: {str(e)}")
                return None
            
            # Analyze the question type to customize prompt
            question_lower = question.lower()
            
            # Determine what type of question is being asked
            is_color_question = any(word in question_lower for word in ["color", "colored", "colors", "red", "blue", "green", "yellow", "black", "white"])
            is_material_question = any(word in question_lower for word in ["material", "made of", "wood", "plastic", "metal", "fabric"])
            is_yesno_question = any(word in question_lower for word in ["is", "are", "does", "has", "can", "do"]) and not any(word in question_lower for word in ["what", "which", "how", "where"])
            is_object_question = any(word in question_lower for word in ["what", "identify", "object", "item", "product"])
            
            # Craft a specific prompt based on question type
            if is_color_question:
                prompt = f"Question: {question} Answer with just the color name in a single word."
            elif is_material_question:
                prompt = f"Question: {question} Answer with just the material name in a single word."
            elif is_yesno_question:
                prompt = f"Question: {question} Answer with only 'yes' or 'no'."
            elif is_object_question:
                prompt = f"Question: {question} Name this object in 1-2 words maximum. No sentences."
            else:
                # Generic prompt for other questions
                prompt = f"Question: {question} Answer with a single word or very short phrase. No sentences."
            
            # Process inputs with improved prompt
            try:
                inputs = self.processor(
                    images=image,
                    text=prompt,
                    return_tensors="pt"
                ).to(self.device)
                logger.debug("Inputs processed successfully")
            except Exception as e:
                logger.error(f"Failed to process inputs: {str(e)}")
                return None
            
            # Generate answer with improved parameters
            try:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=16,      # Keep short but allow enough for meaningful responses
                    num_beams=5,            # More beams for better quality
                    early_stopping=True,
                    do_sample=False,        # Deterministic for consistency
                    # min_length=1,           # Allow single-word answers
                    # max_length=20,          # Hard limit to prevent long sentences
                    repetition_penalty=1.5, # Prevent repetitions
                    length_penalty=0.6      # Slightly favor shorter responses
                )
                logger.debug("Generated output successfully")
            except Exception as e:
                logger.error(f"Failed to generate answer: {str(e)}")
                return None
            
            # Enhanced answer processing
            try:
                raw_answer = self.processor.decode(outputs[0], skip_special_tokens=True)
                logger.debug(f"Raw answer: {raw_answer}")
                
                if not raw_answer:
                    logger.warning("Empty answer generated")
                    return None
                
                # Clean up the answer
                # Convert to lowercase and strip
                processed_answer = raw_answer.lower().strip()
                
                # Remove common prefixes that shouldn't be in the answer
                prefixes_to_remove = ["answer:", "the answer is", "i would say", "it is", "this is", "the object is",
                                   "this object is", "the product is", "a ", "an ", "the ", "it's ", "its ",
                                   "question:", "is the", "is it", "the main", "the color is", "the color of"]
                
                for prefix in prefixes_to_remove:
                    if processed_answer.startswith(prefix):
                        processed_answer = processed_answer[len(prefix):].strip()
                
                # Remove common suffixes that shouldn't be in the answer
                suffixes_to_remove = ["."]
                for suffix in suffixes_to_remove:
                    if processed_answer.endswith(suffix):
                        processed_answer = processed_answer[:-len(suffix)].strip()
                
                # Replace full sentences with key terms
                # For phone cases
                if re.search(r'(phone|galaxy|samsung|lg|iphone|case|cover)', processed_answer):
                    if "case" not in processed_answer and "cover" not in processed_answer:
                        processed_answer = "phone case"
                    else:
                        processed_answer = re.sub(r'.*?(phone case|phone cover|case|cover).*', r'\1', processed_answer)
                
                # For yes/no questions - strictly normalize to just "yes" or "no"
                if is_yesno_question:
                    if any(word in processed_answer for word in ["yes", "yeah", "correct", "true", "affirmative"]):
                        processed_answer = "yes"
                    elif any(word in processed_answer for word in ["no", "not", "negative", "false", "isn't", "isn't"]):
                        processed_answer = "no"
                
                # For color questions - extract just the color
                if is_color_question:
                    color_words = ["red", "blue", "green", "yellow", "black", "white", "orange", "purple", 
                                 "pink", "brown", "gray", "grey", "teal", "navy", "gold", "silver", 
                                 "multicolor", "multicolored", "multi-color", "bronze", "azure", "lilac"]
                    
                    for color in color_words:
                        if color in processed_answer:
                            processed_answer = color
                            break
                
                # For better handling of object identification
                # Map some specific terms to match ground truth patterns
                term_mapping = {
                    "phone case": "cellular phone case",
                    "phone cover": "cellular phone case",
                    "cellphone case": "cellular phone case",
                    "mobile case": "cellular phone case",
                    "mobile phone case": "cellular phone case",
                    "smartphone case": "cellular phone case",
                    "cell case": "cellular phone case",
                    "door hinge": "hardware_hinge",
                    "hinge": "hardware_hinge",
                    "multiple colors": "multicolor",
                    "multi colored": "multicolor",
                    "colorful": "multicolor",
                    "rectangle": "rectangular",
                }
                
                # Apply term mapping
                for term, replacement in term_mapping.items():
                    if term in processed_answer:
                        processed_answer = replacement
                        break
                
                # Limit to 1-2 words for clarity
                words = processed_answer.split()
                if len(words) > 2 and not is_yesno_question:
                    # Keep only the most important 1-2 words
                    important_words = []
                    for word in words:
                        if word not in ["a", "an", "the", "is", "are", "of", "with", "and"]:
                            important_words.append(word)
                            if len(important_words) >= 2:
                                break
                    
                    processed_answer = " ".join(important_words) if important_words else words[0]
                
                # Final cleanup
                processed_answer = processed_answer.strip()
                
                # If answer is empty after processing, return a fallback
                if not processed_answer:
                    # Default based on question type
                    if is_yesno_question:
                        return "yes"  # Most yes/no questions in your dataset seem to be "yes"
                    elif is_color_question:
                        return "black"  # Most common color in many datasets
                    else:
                        return "phone case"  # Common object in your dataset
                
                logger.debug(f"Final processed answer: {processed_answer}")
                return processed_answer
                
            except Exception as e:
                logger.error(f"Failed to decode/process answer: {str(e)}")
                return None
            
        except Exception as e:
            logger.error(f"Inference failed on device {self.device_id}: {str(e)}")
            return None
        finally:
            # Clean up to prevent memory leaks
            clear_memory(self.device_id)
    
    def __del__(self):
        """Clean up resources when model is deleted"""
        try:
            logger.debug("Cleaning up model resources")
            del self.model
            del self.processor
            clear_memory(self.device_id)
        except Exception as e:
            logger.error(f"Error during model cleanup: {str(e)}")

In [37]:
def evaluate_from_csv(csv_path, sample_size=50, base_image_dir="", batch_size=1):
    """Enhanced evaluation workflow with improved resource handling, debugging and comprehensive metrics"""
    start_time = time.time()
    logger.info(f"Starting evaluation with sample size {sample_size}")
    
    # First, check if the CSV file exists
    if not os.path.exists(csv_path):
        logger.error(f"CSV file not found: {csv_path}")
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    
    # Check if the image directory exists
    if base_image_dir and not os.path.exists(base_image_dir):
        logger.error(f"Image directory not found: {base_image_dir}")
        raise FileNotFoundError(f"Image directory not found: {base_image_dir}")
    
    # Check GPU availability
    gpu_count, gpu_memory = get_gpu_info()
    logger.info(f"Found {gpu_count} GPUs with memory: {gpu_memory}")
    
    # If no GPUs are available, use CPU
    if gpu_count == 0:
        logger.warning("No GPUs available, falling back to CPU")
        gpu_count = 1  # Treat it as one "device" (CPU)
    
    try:
        # Load and validate data
        try:
            data = pd.read_csv(csv_path)
            logger.info(f"Loaded CSV with {len(data)} entries")
            
            # Show sample data for debugging
            logger.debug(f"CSV columns: {data.columns.tolist()}")
            logger.debug(f"First few rows: {data.head(2).to_dict('records')}")
            
        except Exception as e:
            logger.error(f"Failed to load CSV: {str(e)}")
            raise
        
        # Sample data
        if sample_size > 0 and sample_size < len(data):
            eval_data = data.sample(sample_size, random_state=42)  # Fixed seed for reproducibility
            logger.info(f"Sampled {len(eval_data)} entries for evaluation")
        else:
            eval_data = data
            logger.info(f"Using all {len(eval_data)} entries for evaluation")
        
        # Validate image paths
        valid_data = []
        invalid_count = 0
        
        for idx, row in eval_data.iterrows():
            try:
                # Check required columns
                if not all(col in row for col in ['path', 'generated_question', 'generated_answer']):
                    missing_cols = [col for col in ['path', 'generated_question', 'generated_answer'] if col not in row]
                    logger.warning(f"Row {idx} missing required columns: {missing_cols}")
                    invalid_count += 1
                    continue
                
                # Build image path
                img_path = os.path.join(base_image_dir, row['path'])
                
                # Check if image exists
                if os.path.exists(img_path):
                    valid_data.append((
                        idx,
                        img_path,
                        row['generated_question'], 
                        row['generated_answer']
                    ))
                else:
                    logger.warning(f"Image not found: {img_path}")
                    invalid_count += 1
            except Exception as e:
                logger.warning(f"Failed to process row {idx}: {str(e)}")
                invalid_count += 1
        
        logger.info(f"Found {len(valid_data)} valid entries with existing images")
        logger.info(f"Skipped {invalid_count} invalid entries")
        
        if not valid_data:
            logger.error("No valid data to process")
            return None
        
        # Use a very small subset for testing if too many items
        if len(valid_data) > 100:
            logger.warning(f"Large dataset with {len(valid_data)} items, using only first 5 for initial testing")
            test_data = valid_data[:5]
            
            # Try to process just a few items first
            logger.info("Testing with a small subset first...")
            
            # Initialize model
            try:
                model = VQAModel(0 if gpu_count > 0 else -1)  # Use first GPU or CPU
                
                # Process test items
                test_results = []
                for item in test_data:
                    try:
                        idx, img_path, question, answer = item
                        prediction = model.infer(img_path, question)
                        test_results.append((idx, prediction, answer))
                    except Exception as e:
                        logger.error(f"Test processing failed: {str(e)}")
                
                # Check if test was successful
                if not any(r[1] is not None for r in test_results):
                    logger.error("Initial test failed to produce any valid predictions")
                    return None
                
                # If test was successful, proceed with full dataset
                logger.info("Initial test successful, proceeding with full dataset")
                del model
                clear_memory()
                
            except Exception as e:
                logger.error(f"Initial test failed: {str(e)}")
                return None
        
        # Process all valid data
        results = []
        
        # Process on CPU if no GPUs or only one item
        if gpu_count == 0 or len(valid_data) <= 1:
            logger.info("Processing on CPU")
            
            try:
                model = VQAModel(-1)  # -1 indicates CPU
                
                for item in tqdm(valid_data, desc="CPU processing"):
                    try:
                        idx, img_path, question, answer = item
                        prediction = model.infer(img_path, question)
                        results.append((idx, prediction, answer))
                        logger.debug(f"Processed item {idx}: Q={question}, A={answer}, P={prediction}")
                    except Exception as e:
                        logger.error(f"Failed to process item {item[0]}: {str(e)}")
                        results.append((item[0], None, item[3]))
                
                del model
                
            except Exception as e:
                logger.error(f"CPU processing failed: {str(e)}")
        
        else:
            # Round-robin distribute items across GPUs
            gpu_items = {i: [] for i in range(gpu_count)}
            
            for i, item in enumerate(valid_data):
                gpu_idx = i % gpu_count
                gpu_items[gpu_idx].append(item)
            
            # Process items on each GPU
            for gpu_idx in range(gpu_count):
                if not gpu_items[gpu_idx]:
                    continue
                    
                logger.info(f"Processing {len(gpu_items[gpu_idx])} items on GPU {gpu_idx}")
                
                try:
                    # Initialize model on this GPU
                    model = VQAModel(gpu_idx)
                    
                    # Process all items for this GPU
                    for item in tqdm(gpu_items[gpu_idx], desc=f"GPU {gpu_idx} processing"):
                        try:
                            idx, img_path, question, answer = item
                            prediction = model.infer(img_path, question)
                            results.append((idx, prediction, answer))
                            logger.debug(f"Processed item {idx}: Q={question}, A={answer}, P={prediction}")
                        except Exception as e:
                            logger.error(f"Failed to process item {item[0]}: {str(e)}")
                            results.append((item[0], None, item[3]))
                        finally:
                            # Give the GPU a moment to recover
                            time.sleep(0.1)
                    
                    # Clean up model when done with this GPU
                    del model
                    clear_memory(gpu_idx)
                    
                except Exception as e:
                    logger.error(f"Processing failed on GPU {gpu_idx}: {str(e)}")
                    # Add empty results for failed items
                    for item in gpu_items[gpu_idx]:
                        results.append((item[0], None, item[3]))
                    
                    # Clean up memory
                    clear_memory(gpu_idx)
        
        # Sort results by index
        results.sort(key=lambda x: x[0])
        
        # Print some raw results for debugging
        logger.debug(f"Raw results (first 5): {results[:5]}")
        
        # Count successful predictions
        successful_preds = sum(1 for _, p, _ in results if p is not None)
        logger.info(f"Generated {successful_preds} valid predictions out of {len(results)} items")
        
        # Filter valid predictions
        valid_results = [(p, t) for _, p, t in results if p is not None and t is not None]
        
        if not valid_results:
            logger.error("No valid predictions generated")
            return None
            
        # Get predictions and ground truths
        predictions, ground_truths = zip(*valid_results)
        
        # Preprocess for evaluation
        clean_predictions = [preprocess_answer(p) for p in predictions]
        clean_ground_truths = [preprocess_answer(t) for t in ground_truths]
        
        # Remove empty strings after preprocessing
        valid_pairs = [(p, t) for p, t in zip(clean_predictions, clean_ground_truths) 
                      if p and t]
        
        if valid_pairs:
            valid_preds, valid_truths = zip(*valid_pairs)
        else:
            logger.warning("No valid prediction-reference pairs after preprocessing")
            valid_preds, valid_truths = [], []
        
        # Calculate traditional metrics
        exact_matches = sum(1 for p, t in zip(clean_predictions, clean_ground_truths) 
                          if p and t and p == t)
        
        partial_matches = sum(1 for p, t in zip(clean_predictions, clean_ground_truths)
                            if p and t and (p in t or t in p))
        
        # Calculate BLEU score
        bleu_score = 0.0
        if valid_pairs:
            bleu_score = compute_bleu_score(valid_preds, valid_truths)
            logger.info(f"BLEU Score: {bleu_score:.4f}")
        
        # Calculate BERTScore
        bert_scores = {"precision": 0.0, "recall": 0.0, "f1": 0.0}
        if valid_pairs:
            bert_scores = compute_bert_score(valid_preds, valid_truths)
            logger.info(f"BERTScore: P={bert_scores['precision']:.4f}, R={bert_scores['recall']:.4f}, F1={bert_scores['f1']:.4f}")
        
        # Create metrics dictionary
        metrics = {
            'exact_match': exact_matches / len(valid_results) if valid_results else 0,
            'partial_match': partial_matches / len(valid_results) if valid_results else 0,
            'bleu': bleu_score,
            'bert_score': bert_scores,
            'valid_predictions': len(valid_pairs),
            'total_predictions': len(valid_results),
            'details': list(zip(predictions, ground_truths))
        }
        
        # Save results to CSV with more detail
        try:
            results_df = pd.DataFrame({
                'prediction': [r[1] for r in results],
                'clean_prediction': [preprocess_answer(r[1]) if r[1] is not None else None for r in results],
                'ground_truth': [r[2] for r in results],
                'clean_ground_truth': [preprocess_answer(r[2]) if r[2] is not None else None for r in results],
                'original_index': [r[0] for r in results],
                'exact_match': [(preprocess_answer(r[1]) == preprocess_answer(r[2])) 
                               if r[1] is not None and r[2] is not None else False for r in results]
            })
            results_df.to_csv("enhanced_results.csv", index=False)
            logger.info("Saved results to enhanced_results.csv")
        except Exception as e:
            logger.error(f"Failed to save results: {str(e)}")
        
        # Log elapsed time
        elapsed = time.time() - start_time
        logger.info(f"Evaluation completed in {elapsed:.2f} seconds")
        
        return metrics
    
    except Exception as e:
        logger.error(f"Evaluation failed: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return None
    
    finally:
        # Final cleanup
        clear_memory()

In [38]:
def main():
    """Main function with simplified approach for testing"""
    # Check filesystem first
    logger.info("Checking filesystem...")
    
    csv_path = "/kaggle/input/new-data/simplified_vqa_dataset.csv"
    image_dir = "/kaggle/input/working-vr/abo-images-small/images/small"
    
    # Check if paths exist
    logger.info(f"CSV path exists: {os.path.exists(csv_path)}")
    logger.info(f"Image dir exists: {os.path.exists(image_dir)}")
    
    # Modify paths if needed for testing
    if not os.path.exists(csv_path):
        csv_path = input("Enter path to CSV file: ")
    
    if not os.path.exists(image_dir):
        image_dir = input("Enter path to image directory: ")
    
    # Run evaluation with very small sample for initial test
    try:
        results = evaluate_from_csv(
            csv_path=csv_path,
            base_image_dir=image_dir,
            sample_size=5000,  # Increased sample size for better evaluation
            batch_size=1     # Process one item at a time
        )
        
        if results:
            print("\n=== Evaluation Results ===")
            print(f"Exact Match: {results['exact_match']:.2%}")
            print(f"Partial Match: {results['partial_match']:.2%}")
            print(f"BLEU Score: {results['bleu']:.4f}")
            print(f"BERTScore: P={results['bert_score']['precision']:.4f}, " + 
                  f"R={results['bert_score']['recall']:.4f}, F1={results['bert_score']['f1']:.4f}")
            print(f"Valid Predictions: {results['valid_predictions']} out of {results['total_predictions']}")
            
            # Show some examples
            print("\nExample predictions (first 5):")
            for i, (pred, truth) in enumerate(results['details'][:5]):
                clean_pred = preprocess_answer(pred) if pred else "None"
                clean_truth = preprocess_answer(truth) if truth else "None"
                match = "✓" if clean_pred == clean_truth else "✗"
                print(f"{i+1}. Prediction: {pred} → {clean_pred}, Ground Truth: {truth} → {clean_truth} {match}")
        else:
            print("Evaluation failed to produce valid results")
    
    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
    
    print("Process completed!")

# calling the main function
main()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GPU 0 processing: 100%|██████████| 2500/2500 [37:00<00:00,  1.13it/s]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Computing BERTScore: 100%|██████████| 157/157 [00:02<00:00, 78.13it/s]



=== Evaluation Results ===
Exact Match: 34.45%
Partial Match: 47.42%
BLEU Score: 0.0880
BERTScore: P=0.6766, R=0.6766, F1=0.6766
Valid Predictions: 4998 out of 4998

Example predictions (first 5):
1. Prediction: acetate → acetate, Ground Truth: silicon → silicon ✗
2. Prediction: cellular phone → cellular phone, Ground Truth: Hello kitty → hello kitty ✗
3. Prediction: cellular phone → cellular phone, Ground Truth: cellular_phone_case → cellular_phone_case ✗
4. Prediction: acetate → acetate, Ground Truth: plastic → plastic ✗
5. Prediction: what type → what type, Ground Truth: Cellular phone case → cellular phone case ✗
Process completed!
